# Beamtime quick-check: integrate one frame and look at it

Drop in a MIDAS parameter file and one detector frame (TIFF or HDF5),
and get back a 2D cake (R × η) plus a 1D radial profile in seconds.

Resolves [issue #36](https://github.com/marinerhemant/MIDAS/issues/36):
a no-compile, drop-in integrator for visual checks during beamtime.

**Edit the paths in cell 2, then *Run All*.**

```bash
pip install midas-integrate
```

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch

from midas_integrate import (
    build_csr,
    integrate,
    load_map,
    parse_params,
    profile_1d,
)
from midas_integrate.detector_mapper import build_and_write_map
from midas_integrate.image import load_image
from midas_integrate.kernels import r_axis, eta_axis

## 1. Inputs — edit these

In [ ]:
param_file = Path('params.txt')          # MIDAS-format parameter file
image_file = Path('frame_00001.tif')     # .tif / .tiff / .h5 / .hdf5

# Only used when image_file is HDF5:
h5_dataset = '/exchange/data'            # path to the dataset inside the file
h5_frame_index = 0                       # which frame to pull from a stack

# Optional dark frame (single 2D image, same shape as data). None to skip.
dark_file = None

# Integration knobs
device = 'cpu'                           # 'cpu', 'cuda', 'cuda:0', 'mps'
mode = 'bilinear'                        # 'floor' (fast) | 'bilinear' (accurate) | 'gradient'
dtype = torch.float32

## 2. Parse the parameter file

In [ ]:
params = parse_params(param_file)
params.validate()

print(f'Detector  : {params.NrPixelsY} x {params.NrPixelsZ} px @ {params.pxY} um')
print(f'Lsd       : {params.Lsd:.1f} um')
print(f'BC        : ({params.BC_y:.2f}, {params.BC_z:.2f}) px')
print(f'Tilts     : tx={params.tx} ty={params.ty} tz={params.tz} deg')
print(f'R bins    : {params.n_r_bins}  ({params.RMin} -> {params.RMax}, step {params.RBinSize})')
print(f'Eta bins  : {params.n_eta_bins} ({params.EtaMin} -> {params.EtaMax}, step {params.EtaBinSize})')

## 3. Build (or reuse) the detector map

`Map.bin` / `nMap.bin` describe how every detector pixel maps into the
(R, η) bin grid. They depend only on geometry, not on the image, so we
build them once per parameter file and cache them next to it.

In [ ]:
map_dir = param_file.parent
map_path = map_dir / 'Map.bin'
nmap_path = map_dir / 'nMap.bin'

if not (map_path.exists() and nmap_path.exists()):
    print('Building detector map (one-time, slow)...')
    build_and_write_map(params, output_dir=map_dir, n_jobs=-1, verbose=True)
else:
    print(f'Reusing cached map at {map_path}')

pixmap = load_map(map_path, nmap_path)
geom = build_csr(
    pixmap,
    n_r=params.n_r_bins, n_eta=params.n_eta_bins,
    n_pixels_y=params.NrPixelsY, n_pixels_z=params.NrPixelsZ,
    bc_y=params.BC_y, bc_z=params.BC_z,
    device=device, dtype=dtype,
    build_modes=(mode,),
)

## 4. Load the frame (TIFF or HDF5)

In [ ]:
def load_frame(path: Path) -> np.ndarray:
    suffix = path.suffix.lower()
    if suffix in ('.h5', '.hdf5', '.nx', '.nxs'):
        with h5py.File(path, 'r') as f:
            dset = f[h5_dataset]
            arr = dset[h5_frame_index] if dset.ndim == 3 else dset[...]
        if arr.shape != (params.NrPixelsZ, params.NrPixelsY):
            raise ValueError(
                f'HDF5 frame shape {arr.shape} != expected '
                f'({params.NrPixelsZ}, {params.NrPixelsY})'
            )
        return np.ascontiguousarray(arr)
    return load_image(path,
                      n_pixels_y=params.NrPixelsY,
                      n_pixels_z=params.NrPixelsZ)

image = load_frame(image_file).astype(np.float64)
if dark_file is not None:
    image = image - load_frame(Path(dark_file)).astype(np.float64)

print(f'Loaded {image_file.name}: shape={image.shape}, '
      f'min={image.min():.1f}, max={image.max():.1f}')

## 5. Integrate

In [ ]:
img_t = torch.from_numpy(image).to(device=device, dtype=dtype)

int2d = integrate(img_t, geom, mode=mode, normalize=bool(params.Normalize))
prof = profile_1d(int2d, geom, mode='area_weighted')

r_vals = r_axis(n_r=params.n_r_bins, RMin=params.RMin, RBinSize=params.RBinSize)
eta_vals = eta_axis(n_eta=params.n_eta_bins,
                    EtaMin=params.EtaMin, EtaBinSize=params.EtaBinSize)

int2d_np = int2d.detach().cpu().numpy()
prof_np = prof.detach().cpu().numpy()

## 6. Look at it

Top: 2D cake (η along x, R along y) on a log scale. Sharp powder rings
show up as horizontal bands; single-crystal spots are localised dots.

Bottom: azimuthally-averaged 1D profile.

In [ ]:
fig, (ax_cake, ax_1d) = plt.subplots(
    2, 1, figsize=(10, 8),
    gridspec_kw={'height_ratios': [2, 1]},
)

vmin = max(np.percentile(int2d_np[int2d_np > 0], 1), 1.0) if (int2d_np > 0).any() else 1.0
vmax = np.percentile(int2d_np, 99.5) if int2d_np.size else 1.0
im = ax_cake.imshow(
    int2d_np, origin='lower', aspect='auto',
    extent=[eta_vals[0], eta_vals[-1], r_vals[0], r_vals[-1]],
    norm=plt.matplotlib.colors.LogNorm(vmin=vmin, vmax=max(vmax, vmin * 10)),
    cmap='viridis',
)
ax_cake.set_xlabel(r'$\eta$ (deg)')
ax_cake.set_ylabel('R (px)')
ax_cake.set_title(f'Cake plot — {image_file.name}')
fig.colorbar(im, ax=ax_cake, label='Intensity (log)')

ax_1d.plot(r_vals, prof_np, lw=1.0)
ax_1d.set_xlabel('R (px)')
ax_1d.set_ylabel('Mean intensity')
ax_1d.set_yscale('log')
ax_1d.set_xlim(r_vals[0], r_vals[-1])
ax_1d.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 7. (Optional) save the lineout

Writes interleaved `(R, I)` float64 pairs — same on-disk layout as the
C `lineout.bin` so downstream MIDAS tools can consume it unchanged.

In [ ]:
out_path = image_file.with_suffix('.lineout.bin')
pairs = np.empty(params.n_r_bins * 2, dtype=np.float64)
pairs[0::2] = r_vals
pairs[1::2] = prof_np.astype(np.float64)
out_path.write_bytes(pairs.tobytes())
print(f'wrote {out_path}')